## Sustainability & Spatial Misalignment in Early-Stage Corridor Development

| Hypothesis | Statement |
|---|---|
| H1 | Built-up expansion is associated with declining vegetation cover and increased surface heat susceptibility |
| H2 | A share of new built-up areas overlaps with environmentally sensitive or water-prone zones |
| H3 |  Areas with low economic utilisation disproportionately coincide with higher environmental exposure |

In [ ]:
import geemap
import ee 
import geemap.colormaps as cm
import os

In [ ]:
geemap.ee_initialize()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

Map = geemap.Map()

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')
Map.centerObject(roi, 10)
Map

In [ ]:
# ── Sentinel-2: 2025 (Oct–Dec post-monsoon composite) ───────
s2_2025 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

# ── Sentinel-2: 2016 (Oct–Dec post-monsoon composite) ───────
s2_2016 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2016-10-01', '2016-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

Computes NDBI, MNDWI, SAVI, and NDVI for a given Sentinel-2 image.  
NDVI is added here (not in RQ1/RQ2) as it provides a cleaner vegetation-loss signal alongside SAVI.

In [ ]:
def calculate_indices(image):
    """
    Computes NDBI, MNDWI, SAVI, NDVI from a Sentinel-2 SR image.
    Band references:
        B2=Blue, B3=Green, B4=Red, B8=NIR, B11=SWIR1
    """
    ndbi = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)',
        {'SWIR1': image.select('B11'), 'NIR': image.select('B8')}
    ).rename('NDBI')

    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)',
        {'Green': image.select('B3'), 'SWIR1': image.select('B11')}
    ).rename('MNDWI')

    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)',
        {'NIR': image.select('B8'), 'Red': image.select('B4')}
    ).rename('SAVI')

    return image.addBands([ndbi, mndwi, savi])


processed_2025 = calculate_indices(s2_2025)
processed_2016 = calculate_indices(s2_2016)

print("Spectral indices computed for 2016 and 2025.")

---
## Built-up Masks (Reusing RQ1 thresholds)

| Year | NDBI threshold | Rationale |
|---|---|---|
| 2025 | > 0.05 | Post-monsoon; lower saline soil noise |
| 2016 | > 0.13 | Higher radiometric contrast from saline pans in drier 2016 season |

In [ ]:
# ── 2025 Built-up Mask ───────────────────────────────────────
built_up_2025 = processed_2025.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2025.select('NDBI'),
        'MNDWI': processed_2025.select('MNDWI'),
        'SAVI':  processed_2025.select('SAVI')
    }
).rename('BuiltUp_2025')

# ── 2016 Built-up Mask ───────────────────────────────────────
built_up_2016 = processed_2016.expression(
    '(NDBI > 0.13) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2016.select('NDBI'),
        'MNDWI': processed_2016.select('MNDWI'),
        'SAVI':  processed_2016.select('SAVI')
    }
).rename('BuiltUp_2016')

print("Built-up masks ready: 2016 and 2025.")

---
### Land Transformation Map

A two-panel view of:
- **Built-up change** (2016 → 2025) — reusing the 4-class growth heatmap from RQ1
- **Vegetation loss** — NDVI/SAVI change between 2016 and 2025

Both layers are shown together so you can read **where built-up growth co-occurred with vegetation loss**.

Built-up Growth Heatmap (2016 → 2025)

| Value | Class | Colour |
|---|---|---|
| 0 | No built-up (both years) | ⬛ |
| 1 | Stable built-up | ⬜ |
| 2 | New growth 2016→2025 | 🟠 |
| 3 | Lost built-up | 🔵 |

In [ ]:
# ── Built-up Change Classification ──────────────────────────
change_stack = built_up_2016.rename('y2016').addBands(built_up_2025.rename('y2025'))

growth_class = change_stack.expression(
    "(b('y2016') == 0 && b('y2025') == 0) ? 0"   # No built-up
    ": (b('y2016') == 1 && b('y2025') == 1) ? 1"  # Stable
    ": (b('y2016') == 0 && b('y2025') == 1) ? 2"  # New Growth
    ": (b('y2016') == 1 && b('y2025') == 0) ? 3"  # Lost
    ": 0"
).rename('GrowthClass').clip(roi)

### Vegetation Loss Layer (NDVI change & SAVI change)

Negative delta = vegetation lost.  
We compute both NDVI delta and SAVI delta - SAVI is more reliable over Dholera's arid/saline background.

In [ ]:
# ── Vegetation Change Layers ─────────────────────────────────

# SAVI delta: more soil-noise-robust over arid landscapes
savi_2025 = processed_2025.select('SAVI')
savi_2016 = processed_2016.select('SAVI')
savi_delta = savi_2025.subtract(savi_2016).rename('SAVI_Delta').clip(roi)

print("Vegetation change layers computed.")

#### Visualising SAVI and change in SAVI (2025-2016)

In [ ]:

# --- Map 3: SAVI (Testing)---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
test_params = {'min': 0, 'max': 1, 'palette': ['8b0000', 'f5f5f5', '006400']}

Map_Test.addLayer(savi_2025, test_params, 'SAVI (2025)')
Map_Test.addLayer(savi_2016, test_params, 'SAVI (2016)')
Map_Test.add_colorbar(vis_params=test_params, label="Test Value", orientation="horizontal")
print("Displaying Test Map...")
display(Map_Test)

In [ ]:
# --- Delta Calculations ---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
# SAVI Delta: (Later Year - Earlier Year)
# Positive values = Increase in vegetation/soil health
# Negative values = Decrease/Degradation
savi_delta = savi_2025.subtract(savi_2016)

# Define a specific Delta Palette (Red for decrease, White for neutral, Blue/Green for increase)
delta_params = {'min': -0.5, 'max': 0.5, 'palette': ['8b0000', 'f5f5f5', '006400']}

# Add to a new map or the existing one
Map_Test.addLayer(savi_delta, delta_params, 'SAVI Change (2016-2025)')

Map_Test.add_colorbar(vis_params=delta_params, label="Delta Value", orientation="horizontal")

print("Delta layers calculated and added.")
display(Map_Test)

It is observed that net SAVI of 2025 is actually higher in the whole region of Dholera, because of higher monsoon that year in 2025 compared to 2016 in the same time period of (Oct-Dec). Hence the vegetation cover of 2025 in total is actually higher. 
 
That's why we specifically choose to check change in Savi in the region of active construction. This will give us an accurate picture of how much vegetation is lost due to vegetation.

#### OUTPUT 1 : Visualise: Land Transformation Map

In [ ]:
# ── Output 1: Built-up Transformation Map ────────────────────
# Logic: Growth Class == 2 (New Growth) AND SAVI Delta < 0
# This isolates pixels that urbanised AND lost vegetation simultaneously

# Step 1: Mask SAVI delta to new growth footprint only
new_growth_mask   = growth_class.eq(2)                        # Class 2 = new growth pixels
veg_loss_in_growth = savi_delta.updateMask(                   # Keep only pixels where:
    new_growth_mask                                            #   (a) new growth occurred
    .And(savi_delta.lt(0))                                     #   (b) SAVI declined
)

# Step 2: New growth footprint as a flat base layer (charcoal)
new_growth_flat = new_growth_mask.selfMask()

# ── Map ──────────────────────────────────────────────────────
Map_LT = geemap.Map()
Map_LT.centerObject(roi, 11)

# Reference
Map_LT.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# Layer 1: Full new-growth footprint as flat charcoal base
# This shows the full 34 km² even where SAVI didn't decline
Map_LT.addLayer(
    new_growth_flat,
    {'min': 1, 'max': 1, 'palette': ['444444']},
    '⬛ New Growth Footprint (34 km²)'
)

# Layer 2: Vegetation loss gradient — overlaid WITHIN the new growth footprint
# Palette: pale yellow (marginal loss) → deep red (severe loss)
# min is set to -0.3 instead of -0.5 to stretch the colour ramp
# across the realistic SAVI loss range for Dholera's arid landscape
Map_LT.addLayer(
    veg_loss_in_growth,
   # {'min': -0.3, 'max': 0.0, 'palette': ['ffffb2', 'fd8d3c', 'e31a1c', '800026']},
    {'min': -0.3, 'max': 0.0, 'palette': ['800026','e31a1c','fd8d3c','ffffb2']},
    '🔴 Vegetation Loss Intensity (SAVI Decline in New Growth)'
)

# Colorbar
Map_LT.add_colorbar(
    vis_params={'min': -0.3, 'max': 0.0,
                'palette': ['800026','e31a1c','fd8d3c','ffffb2']},
    label='ΔSAVI (0 = No Loss  →  −0.3 = Severe Loss)',
    orientation='horizontal'
)

Map_LT.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Output 1: Built-up Transformation Map ready.")
display(Map_LT)

### Vegetation Loss Area Statistics

Measures the area where SAVI *declined* (vegetation lost) within the new growth footprint - directly tests **H1**.

In [ ]:
# ── Step 3: Key Statistic — Average SAVI Loss per km² of New Growth ──
# Mean SAVI delta across the masked (new growth AND veg loss) zone
mean_savi_loss = savi_delta.updateMask(new_growth_mask.And(savi_delta.lte(0))) \
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

pixel_area = ee.Image.pixelArea()

# Area of new growth with confirmed veg loss
veg_loss_area_m2 = new_growth_mask.And(savi_delta.lt(0)) \
    .multiply(pixel_area) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

avg_savi_loss   = round(mean_savi_loss.get('SAVI', 0), 4)
veg_loss_km2    = round(list(veg_loss_area_m2.values())[0] / 1e6, 3)
total_growth_km2 = 34.013   # from RQ1

print()
print("=" * 55)
print("  OUTPUT 1 — KEY STATISTICS")
print("=" * 55)
print(f"  🟠 New Built-up Growth (2016→2025)     : {total_growth_km2} km²")
print(f"  🌿 Growth ∩ Vegetation Loss Area        : {veg_loss_km2} km²")
if total_growth_km2 > 0:
    pct = round(veg_loss_km2 / total_growth_km2 * 100, 1)
    print(f"  → {pct}% of new growth occurred at cost of local biomass")
print(f"  📉 Avg SAVI Loss per km² of New Growth  : {avg_savi_loss}")
print()
print("  Ecological Interpretation:")
print("  Each km² of new industrial footprint carries an average SAVI")
print(f"  decline of {avg_savi_loss} - the ecological price tag of corridor expansion.")
print("=" * 55)

---
## OUTPUT 2 : Heat Susceptibility Map

> **Proxy for urban heat stress: high built-up density + low vegetation**

This is a **composite index** - not a direct LST measurement - built from:
- `NDBI` (built-up surface): higher = more heat-absorbing materials
- `SAVI` (vegetation): higher = more evapotranspiration, lower heat stress

**Formula:**
```
Heat_Index = NDBI_norm - SAVI_norm
```
Values range ~[-1 to +1].  
+ Positive = high heat susceptibility (built-up, no vegetation)  
- Negative = low heat susceptibility (vegetated, no built-up)

In [ ]:
# ── Heat Susceptibility Index (2025) ─────────────────────────
# Normalise NDBI and SAVI to 0–1 range before differencing
# Using min–max normalisation anchored to realistic Sentinel-2 ranges

ndbi_raw  = processed_2025.select('NDBI')
savi_raw  = processed_2025.select('SAVI')

# NDBI normalised: range −0.5 to +0.5
ndbi_norm = ndbi_raw.add(0.5).divide(1.0).clamp(0, 1).rename('NDBI_norm')

# SAVI normalised: range 0 to 0.8
savi_norm = savi_raw.divide(0.8).clamp(0, 1).rename('SAVI_norm')

# Heat Susceptibility = Built-up signal minus Vegetation buffer
heat_index = ndbi_norm.subtract(savi_norm).rename('HeatIndex').clip(roi)

print("Heat Susceptibility Index computed.")

In [ ]:
# ── Visualise: Heat Susceptibility Map ───────────────────────
Map_Heat = geemap.Map()
Map_Heat.centerObject(roi, 11)

# Reference
Map_Heat.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# --- Layer 1: Continuous Heat Index (gradient) ---
# Palette: cool blue → white → fiery red
Map_Heat.addLayer(
    heat_index,
    {'min': -0.5, 'max': 0.7, 'palette': ['313695', '74add1', 'ffffbf', 'f46d43', 'a50026']},
    '🌡️ Heat Susceptibility Index (Continuous)'
)

# Colorbar for heat index
Map_Heat.add_colorbar(
    vis_params={'min': -0.5, 'max': 0.7, 'palette': ['313695', '74add1', 'ffffbf', 'f46d43', 'a50026']},
    label='Heat Susceptibility Index  \n (−0.5 = Cool/Vegetated → \n +0.7 = Hot/Built-up)',
    orientation='horizontal'
)

Map_Heat.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Heat Susceptibility Map ready.")
display(Map_Heat)

---
## OUTPUT 3 : Flood Risk Overlay

🗺️ Elevation Model Selection: Copernicus GLO-30
For the Flood Vulnerability Model (RQ3), Copernicus GLO-30 (DSM) was selected over FABDEM and ALOS PALSAR to ensure accuracy in Dholera’s unique coastal terrain:
- **Eliminates Algorithmic Bias:** Unlike FABDEM’s ML-based corrections, GLO-30 relies on direct observations, avoiding misclassification of Dholera’s high-reflectance saline crusts as vegetation or built-up areas.
- **Higher Vertical Reliability:** GLO-30 offers superior vertical precision in open terrain, enabling more accurate identification of low-lying zones (< 5 m) than ALOS PALSAR’s coarser 12.5 m data.
- **Ensures Scientific Rigor:** A sensor-based dataset anchors the Sustainability Misalignment analysis in observed reality, minimizing reliance on modeled or inferred corrections.


In [ ]:

# Define your ROI (same as your Sentinel-2 script)
# roi = ee.Geometry.Rectangle([...])  # <-- your ROI here
# 

# Load Copernicus DEM GLO-30
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
    .select('DEM') \
    .filterBounds(roi) \
    .mosaic() \
    .clip(roi)

# Visualization parameters
vis = {
    "min": 0,
    "max": 12,
    "palette": [
        "#0d0887", "#46039f", "#7201a8", "#9c179e",
        "#bd3786", "#d8576b", "#ed7953", "#fb9f3a",
        "#fdca26", "#f0f921"
    ]
}

# Create map
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(dem, vis, "Copernicus DEM GLO-30")

# Add legend at bottom right
Map.add_colorbar(
    vis_params=vis,
    label="Elevation (m)",
    layer_name="Copernicus DEM GLO-30",
    orientation="horizontal",
    position="bottomright"
)

Map

#### ============================================================
#### OUTPUT 3 - Flood Risk Overlay
#### Bivariate Flood Vulnerability Model (FVI)
#### Sources: Copernicus GLO-30 (terrain) + Sentinel-2 monsoon MNDWI
#### ============================================================

In [ ]:
# ── Step 1: Load Copernicus GLO-30 DEM ───────────────────────
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
    .select('DEM') \
    .filterBounds(roi) \
    .mosaic() \
    .clip(roi) \
    .rename('Elevation')

print("GLO-30 DEM loaded.")

In [ ]:
# ── Step 2: Load Sentinel-2 Monsoon Composite (Peak Monsoon) ─
# June 15 – Sept 30, 2025 — captures active water pooling
# Higher cloud threshold (20%) accepted because monsoon scenes
# are inherently cloudy; median reducer suppresses cloud noise

s2_monsoon = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-06-15', '2025-09-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40)) \
    .median() \
    .clip(roi)

# MNDWI from monsoon composite
# (Green - SWIR1) / (Green + SWIR1)
# Positive = water/saturation, Negative = dry/built-up
mndwi_monsoon = s2_monsoon.normalizedDifference(['B3', 'B11']) \
    .rename('MNDWI_Monsoon') \
    .clip(roi)

print("Monsoon MNDWI computed (Jun 15 – Sep 30, 2025).")

In [ ]:
# ── Step 3: Visualise Inputs Before Classification ───────────
Map_Inputs = geemap.Map()
Map_Inputs.centerObject(roi, 10)

# DEM layer
Map_Inputs.addLayer(
    dem,
    {'min': 0, 'max': 15,
     'palette': ['0d0887', '46039f', '7201a8', '9c179e',
                 'bd3786', 'd8576b', 'ed7953', 'fb9f3a', 'fdca26', 'f0f921']},
    '🏔️ GLO-30 Elevation (m)'
)

# Monsoon MNDWI
Map_Inputs.addLayer(
    mndwi_monsoon,
    {'min': -0.3, 'max': 0.5, 'palette': ['white', 'lightblue', 'blue']},
    '💧 Monsoon MNDWI (Jun–Sep 2025)', shown=False
)

Map_Inputs.add_colorbar(
    vis_params={'min': 0, 'max': 15,
                'palette': ['0d0887', '46039f', '7201a8', '9c179e',
                            'bd3786', 'd8576b', 'ed7953', 'fb9f3a', 'fdca26', 'f0f921']},
    label='Elevation (m)',
    orientation='horizontal'
)

Map_Inputs.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Input layers ready — toggle DEM and MNDWI to inspect before classifying.")
display(Map_Inputs)

In [ ]:
# ── Step 4: Bivariate Flood Vulnerability Classification ─────
#
# Class 3 — 🚨 HIGH FLOOD RISK ("The Sink")
#   Elevation ≤ 5m  AND  MNDWI > 0.1
#   → Lowest topographic depressions + proven monsoon saturation
#
# Class 2 — ⚠️ MODERATE FLOOD RISK ("The Fringe")
#   Elevation ≤ 8m  AND  MNDWI > -0.1  (excluding Class 3 pixels)
#   → Low-lying but with some drainage, OR saturated above the sink line
#
# Class 1 — ✅ LOW FLOOD RISK ("The Resilient Zone")
#   Elevation > 8m  OR  MNDWI < -0.1
#   → Topographically safe or efficiently drained
#
# Class 0 — Background (everything else / masked)

elev  = dem.select('Elevation')
mndwi = mndwi_monsoon.select('MNDWI_Monsoon')

# Build classification using .where() — priority order matters:
# High risk first so moderate doesn't overwrite it
fvi = ee.Image(1) \
    .where(
        elev.lte(8).And(mndwi.gt(-0.1)),
        2                                   # Moderate
    ) \
    .where(
        elev.lte(5).And(mndwi.gt(0.1)),
        3                                   # High — overwrites Moderate where both apply
    ) \
    .rename('FVI') \
    .clip(roi)

print("Flood Vulnerability Index classified.")

In [ ]:
# ── Step 5: Visualise FVI Map ────────────────────────────────
fvi_palette = {
    'min': 1,
    'max': 3,
    'palette': [
        '2d6a4f',   # 1 → Green  = Low Risk
        'f4a261',   # 2 → Amber  = Moderate Risk
        'd62828',   # 3 → Red    = High Risk
    ]
}

Map_FVI = geemap.Map()
Map_FVI.centerObject(roi, 11)

# Reference
Map_FVI.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# FVI classification
Map_FVI.addLayer(fvi, fvi_palette, '🌊 Flood Vulnerability Index (FVI)')

# Overlay new built-up growth for H2 context
# Shows where urbanisation has encroached on flood-prone zones
Map_FVI.addLayer(
    new_growth_flat,
    {'min': 1, 'max': 1, 'palette': ['ffffff']},
    '⬛ New Growth Footprint (2016→2025)', shown=False
)

Map_FVI.add_legend(
    title='Flood Vulnerability Index',
    legend_dict={
        '✅ Low Risk  — Elevation > 8m or MNDWI < −0.1':  '2d6a4f',
        '⚠️ Moderate  — Low-lying or seasonally saturated': 'f4a261',
        '🚨 High Risk — Sink zones (≤5m + MNDWI > 0.1)':  'd62828',
    },
    position='bottomleft'
)

Map_FVI.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Flood Vulnerability Map ready.")
display(Map_FVI)

In [ ]:
# ── Step 6: Area Statistics + H2 Test ───────────────────────
# H2: A share of new built-up areas overlaps with flood-prone zones

pixel_area = ee.Image.pixelArea()

def fvi_area_km2(class_val):
    """Returns area in km² for a given FVI class."""
    mask = fvi.eq(class_val)
    result = mask.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    )
    return round(result.getInfo().get('FVI', 0) / 1e6, 3)

def overlap_km2(fvi_class, growth_mask):
    """Returns area where a FVI class overlaps with a given growth mask."""
    mask = fvi.eq(fvi_class).And(growth_mask)
    result = mask.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    )
    return round(result.getInfo().get('FVI', 0) / 1e6, 3)

# Total area per FVI class
low_km2      = fvi_area_km2(1)
moderate_km2 = fvi_area_km2(2)
high_km2     = fvi_area_km2(3)
total_km2    = low_km2 + moderate_km2 + high_km2

# New growth (Class 2 from growth_class) overlap with flood zones
new_growth_pixels = growth_class.eq(2)
growth_in_moderate = overlap_km2(2, new_growth_pixels)
growth_in_high     = overlap_km2(3, new_growth_pixels)
growth_in_risk     = round(growth_in_moderate + growth_in_high, 3)
total_growth_km2   = 34.013   # from RQ1

print("=" * 60)
print("  OUTPUT 3 — FLOOD VULNERABILITY REPORT")
print("=" * 60)
print(f"  ✅ Low Risk Zone      : {low_km2:>8.3f} km²  ({low_km2/total_km2*100:>5.1f}%)")
print(f"  ⚠️  Moderate Risk Zone : {moderate_km2:>8.3f} km²  ({moderate_km2/total_km2*100:>5.1f}%)")
print(f"  🚨 High Risk Zone     : {high_km2:>8.3f} km²  ({high_km2/total_km2*100:>5.1f}%)")
print("-" * 60)
print(f"  📦 Total ROI Area     : {total_km2:>8.3f} km²")
print()
print("  H2 — New Built-up Growth ∩ Flood Risk Zones:")
print(f"  ⚠️  Growth in Moderate Risk  : {growth_in_moderate:>7.3f} km²")
print(f"  🚨 Growth in High Risk       : {growth_in_high:>7.3f} km²")
print(f"  → Total Growth in Risk Zones : {growth_in_risk:>7.3f} km²")
if total_growth_km2 > 0:
    pct_risk = round(growth_in_risk / total_growth_km2 * 100, 1)
    print(f"  → {pct_risk}% of new built-up growth sits inside flood-vulnerable zones.")
print()
print("  Interpretation:")
print("  Infrastructure deployment ahead of urbanisation has placed a")
print("  measurable share of Dholera's new footprint inside terrain that")
print("  is topographically low-lying and seasonally water-saturated —")
print("  confirming H2 and adding a flood exposure dimension to the")
print("  spatial misalignment argument.")
print("=" * 60)